# Healthcare Cost Prediction

**Problem statement:** A healthcare planning analyst wants a rough annual-cost estimate from age, BMI, smoking status, and chronic-condition status to explore high-level budget scenarios. This notebook tests whether those fields provide useful signal beyond simply predicting the training-set average cost.

This notebook cleans the supplied data, performs a small exploratory analysis, fits one regression model, and evaluates it on a held-out test set. The dataset is small and should not be used for clinical or pricing decisions.

## 1. Load and inspect the data

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

data_path = Path("healthcare_costs.csv")
if not data_path.exists():
    raise FileNotFoundError(
        f"Could not find {data_path.resolve()}. Place this notebook beside healthcare_costs.csv."
    )

raw_df = pd.read_csv(data_path)
print(f"Rows: {raw_df.shape[0]} | Columns: {raw_df.shape[1]}")
print("Columns:", raw_df.columns.tolist())
display(raw_df.head())
raw_df.info()

## 2. Clean the data

Remove duplicate records and rows without the target. Keep `PatientID` for identification only; it is not a predictive feature. Missing predictor values are imputed inside the modeling pipeline so imputation statistics are learned from the training split only.

## Questions to answer, in order

These are the questions I would ask before building an AI/ML workflow. The code cells below answer them from the supplied data rather than assuming the dataset is complete or that the model will be useful.

1. **Which dataset and decision context?** Use `healthcare_costs.csv` for a preliminary, cohort-level budget-planning exploration—not individual clinical decisions.
2. **What should the model predict, and is this regression or classification?** Predict `AnnualCost`; it is numeric, so this is a regression problem.
3. **What information is available, and what should not be a feature?** Check columns and types; use `Age`, `BMI`, `SmokingStatus`, and `ChronicCondition`. Exclude `PatientID`, since it identifies a row rather than describing patient risk.
4. **Can these records be used as-is?** Check duplicates, missingness, row count, and category/value summaries. The cleaning and profile outputs below show what was actually present and how many rows remain.
5. **What patterns are visible before modeling?** Inspect the target distribution and compare cost summaries across smoking groups. These are descriptive patterns, not evidence of causation.
6. **Does the model add value, and how uncertain is that answer?** Compare held-out MAE, RMSE, and R² with a training-mean benchmark. The test set has only 10 records, so the comparison is a preliminary check, not a reliable estimate of future performance.

In a real deployment, I would also ask who is affected by the estimates, how the target cost is defined, whether the data represents the intended population, what privacy and fairness constraints apply, and what decision the estimate is allowed to influence.

In [ ]:
df = raw_df.copy()
duplicate_count = int(df.duplicated().sum())
missing_before = df.isna().sum()
df = df.drop_duplicates().dropna(subset=["AnnualCost"]).copy()

print(f"Duplicate rows removed: {duplicate_count}")
print(f"Rows with missing AnnualCost removed: {int(missing_before['AnnualCost'])}")
print("Missing values after target cleanup:")
display(df.isna().sum().rename("missing_count").to_frame())
print(f"Cleaned rows: {len(df)}")

In [ ]:
print("Dataset and target check")
print(f"Rows before cleaning: {len(raw_df)}")
print(f"Target: AnnualCost ({raw_df['AnnualCost'].dtype})")
print(f"Candidate input fields: {[column for column in raw_df.columns if column not in ['PatientID', 'AnnualCost']]}")
print(f"PatientID unique: {raw_df['PatientID'].is_unique}")

print("\nData quality check")
print(f"Exact duplicate rows: {int(raw_df.duplicated().sum())}")
display(raw_df.isna().sum().rename("missing_count").to_frame())
print("Numeric field ranges:")
display(raw_df[["Age", "BMI", "AnnualCost"]].agg(["min", "median", "max"]).T)
print("Categorical values:")
for column in ["SmokingStatus", "ChronicCondition"]:
    print(f"{column}: {sorted(raw_df[column].dropna().unique().tolist())}")

print("\nCleaning outcome")
print(f"Rows after duplicate/target cleanup: {len(df)}")

## 3. Exploratory data analysis

Review numeric summaries, missingness, the target distribution, and annual cost by smoking status.

In [ ]:
display(df.describe(include="all").T)
print("Annual cost by smoking status:")
display(
    df.groupby("SmokingStatus")["AnnualCost"]
    .agg(patient_count="size", mean_cost="mean", median_cost="median")
    .round(2)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df["AnnualCost"].dropna(), bins=12, edgecolor="black")
axes[0].set(title="Annual healthcare cost distribution", xlabel="Annual cost", ylabel="Patient count")
df.boxplot(column="AnnualCost", by="SmokingStatus", ax=axes[1])
axes[1].set(title="Annual cost by smoking status", xlabel="Smoking status", ylabel="Annual cost")
fig.suptitle("")
fig.tight_layout()
plt.show()

## 4. Train a regression model

Use an 80/20 train-test split and linear regression, a readable baseline for a numeric target. Numeric inputs are median-imputed; categorical inputs are most-frequent-imputed and one-hot encoded. The patient identifier is excluded. In the evaluation section, compare it with a simple benchmark that always predicts the training-set mean.

In [ ]:
numeric_features = ["Age", "BMI"]
categorical_features = ["SmokingStatus", "ChronicCondition"]
feature_columns = numeric_features + categorical_features
target_column = "AnnualCost"

X = df[feature_columns]
y = df[target_column]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_pipeline = Pipeline(
    steps=[("imputer", SimpleImputer(strategy="median"))]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ]
)
model = Pipeline(
    steps=[("preprocessor", preprocessor), ("regressor", LinearRegression())]
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(f"Training rows: {len(X_train)} | Test rows: {len(X_test)}")

## 5. Evaluate

Compare the feature-based model with a training-mean benchmark that does not use patient features. MAE and RMSE are in the target's cost units; lower is better. A negative R² means the predictions are worse than using the test-set mean as a reference. With only 10 test records, all metrics are unstable and should be interpreted cautiously.

In [ ]:
baseline_predictions = np.full(len(y_test), y_train.mean())

def regression_metrics(actual, predicted):
    return {
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    }

model_metrics = regression_metrics(y_test, predictions)
baseline_metrics = regression_metrics(y_test, baseline_predictions)
metrics = pd.DataFrame(
    [model_metrics, baseline_metrics],
    index=["Linear regression", "Training-mean baseline"],
)
metrics.index.name = "Model"
display(metrics)
print(
    "On this split, linear regression "
    + ("beats" if model_metrics["MAE"] < baseline_metrics["MAE"] else "does not beat")
    + " the training-mean baseline on MAE."
)
display(
    pd.DataFrame(
        {"Actual annual cost": y_test, "Predicted annual cost": predictions}
    ).reset_index(drop=True).head(10)
)

## Limitations

This is a small, likely synthetic sample, so the test metrics are only a demonstration of the workflow and may vary substantially with a different split. The model omits many real-world cost drivers and must not be used to make clinical, coverage, or pricing decisions.